# 06 — Files and Grade Analyzer
**Python Foundations · Slides 39–46 · Python 3.10+**

Run this notebook in Google Colab or local JupyterLab. No extra packages are needed for the lesson code.
Run cells from top to bottom with **Shift+Enter**. Each notebook is independent.
All names and grades are fictional. Predict each result before running its cell.


## Learning goals
Use paths, read and write UTF-8 text, load CSV records, and save an analysis report.

`pathlib` and `csv` are standard-library modules. An `import` makes their tools available. All data are fictional. The illustrative passing threshold is 75.

**Fresh Colab:** sample data are created below if missing. No GitHub download, upload, or Drive mounting is needed. Existing CSV data are preserved. Generated reports are overwritten on repeat runs.


In [ ]:
from pathlib import Path

# JupyterLab may start in the repository root or in notebooks/.
# A fresh Colab session uses its current working directory.
base = Path.cwd()
if base.name == "notebooks":
    base = base.parent
data_dir = base / "data"
output_dir = base / "outputs"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Working folder:", base)


## Text files
`"w"` creates or replaces a file. `"r"` reads an existing file. `"a"` appends. `with` closes the file after its block. `\n` represents a newline. Use UTF-8 explicitly.


In [ ]:
note_path = output_dir / "practice_note.txt"
with note_path.open("w", encoding="utf-8") as file:
    file.write("Python can save our results.\n")
with note_path.open("r", encoding="utf-8") as file:
    message = file.read()
print(message, end="")


**Expected:** `Python can save our results.`


## Complete analyzer functions
This cell is the same implementation used by `grade_analyzer.py`. First follow `ensure_sample_data`, `read_students`, `summarize`, and `write_report`. Validation inside `read_students` is reference material: it gives helpful feedback for invalid grades and CSV headers.

`csv.DictReader` gives one dictionary per row. CSV values arrive as strings. `float()` converts each grade. `newline=""` lets the CSV module handle line endings correctly.


In [ ]:
from pathlib import Path
import csv
import math

PASSING_GRADE = 75  # Illustrative teaching rule, not institutional policy.
SAMPLE_CSV = "name,grade\nMaya,88\nLuis,72\nAri,95\nBea,75\n"


def ensure_sample_data(path):
    """Create fictional sample data only when the file is missing."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", encoding="utf-8", newline="") as file:
            file.write(SAMPLE_CSV)


def read_students(path):
    """Read name/grade CSV records and reject incomplete or invalid data."""
    students = []
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        if reader.fieldnames != ["name", "grade"]:
            raise ValueError("CSV headers must be: name,grade")
        for row_number, row in enumerate(reader, start=2):
            name = (row.get("name") or "").strip()
            try:
                grade = float(row["grade"])
            except (TypeError, ValueError) as error:
                raise ValueError(f"Row {row_number}: grade must be a number") from error
            if not name or None in row or not math.isfinite(grade) or not 0 <= grade <= 100:
                raise ValueError(f"Row {row_number}: use a name and a grade from 0 to 100")
            students.append({"name": name, "grade": grade})
    return students


def summarize(students, passing_grade=PASSING_GRADE):
    """Return a report, including a helpful result for an empty class."""
    if not students:
        return "No student records.\n"
    total = 0
    passing_names = []
    for student in students:
        total += student["grade"]
        if student["grade"] >= passing_grade:
            passing_names.append(student["name"])
    average = total / len(students)
    names = ", ".join(passing_names) if passing_names else "None"
    return (
        f"Students: {len(students)}\n"
        f"Class average: {average:.2f}\n"
        f"Passing threshold (illustrative): {passing_grade}\n"
        f"Passing students: {len(passing_names)}\n"
        f"Names: {names}\n"
    )


def write_report(path, report):
    """Replace the generated report on every run."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        file.write(report)


## Reading student records
The CSV must have `name,grade` headers and grades from 0 to 100. A header-only file represents an empty class. Missing files get fictional sample data.


In [ ]:
csv_path = data_dir / "grades.csv"
ensure_sample_data(csv_path)
students = read_students(csv_path)
for student in students:
    print(student["name"], student["grade"])


**Expected with bundled data:** `Maya 88.0`, `Luis 72.0`, `Ari 95.0`, `Bea 75.0` on separate lines.


## Analysis and writing
The same input must produce the same summary in the notebook and script. The report uses a trailing newline to keep text files readable.


In [ ]:
report = summarize(students)
report_path = output_dir / "grade_summary.txt"
write_report(report_path, report)
print(report, end="")
print("Saved:", report_path)
with report_path.open("r", encoding="utf-8") as file:
    saved_report = file.read()
assert saved_report == report


**Expected with bundled data**
```text
Students: 4
Class average: 82.50
Passing threshold (illustrative): 75
Passing students: 3
Names: Maya, Ari, Bea
```
The final `Saved:` line shows your runtime-specific path.

## Your turn
Write a second report using a threshold of 90. Save it as `outputs/high_threshold_summary.txt`. Predict who passes. Keep the default report unchanged.


In [ ]:
# Write your answer here.


---
## Solution


In [ ]:
high_report = summarize(students, passing_grade=90)
high_path = output_dir / "high_threshold_summary.txt"
write_report(high_path, high_report)
print(high_report, end="")


**Expected with bundled data:** average 82.50, threshold 90, one passing student, Ari.

## Finding and keeping your files
Locally, open the printed `outputs` folder or use JupyterLab's file browser. In Colab, open **Files** in the left sidebar, refresh it, find `outputs`, and use the report's menu to **Download**. Download before the runtime resets because runtime files are temporary. Saving a notebook to Drive does not automatically save these generated files.

## Exit check and optional extension
Explain why rerunning `"w"` does not duplicate the report. Optional: add a new fictional student to a separate CSV and recalculate. A class with no records produces `No student records.`.

## Official references
[Python files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files) · [csv](https://docs.python.org/3/library/csv.html) · [pathlib](https://docs.python.org/3/library/pathlib.html)
